In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Загрузка данных
train_data = pd.read_csv('/content/drive/MyDrive/response_train.csv')
test_data = pd.read_csv('/content/drive/MyDrive/response_test.csv')

In [4]:
train_data

,client_id,target,age,job-status,retrirement-status,gender,children,dependants,education,marital-status,...,reg-phone,work-phone,loans,paid-loans,total-payments,total-of-delinquencies,max-delinquency-no,mean-delinquency-amount,max-delinquency-amount,previous-cards
0,985668,0,41,1,0,1,3,1,Среднее специальное,Состою в браке,...,0,1,1,0,3,0,0,0.0,0.0,NaN
1,1006558,0,29,1,0,1,1,0,Высшее,Разведен(а),...,0,1,1,0,4,0,0,0.0,0.0,NaN
2,1000371,1,27,1,0,1,0,0,Среднее специальное,Не состоял в браке,...,0,1,1,0,8,0,0,0.0,0.0,NaN
3,985967,0,26,1,0,1,0,0,Среднее специальное,Гражданский брак,...,0,1,2,2,7,0,0,0.0,0.0,NaN
4,1003936,1,51,0,1,1,1,1,Среднее специальное,Состою в браке,...,0,0,1,1,3,0,0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25096,987149,0,45,1,0,0,2,2,Среднее специальное,Состою в браке,...,0,1,2,2,18,0,0,0.0,0.0,NaN
25097,987934,0,27,1,0,1,0,0,Среднее специальное,Не состоял в браке,...,0,1,2,1,8,1,1,1160.0,1160.0,NaN
25098,987993,0,62,0,1,0,2,0,Среднее,Не состоял в браке,...,0,0,1,0,4,0,0,0.0,0.0,NaN
25099,1002020,0,44,1,0,0,0,0,Среднее специальное,Состою в браке,...,0,1,1,0,4,0,0,0.0,0.0,NaN


In [5]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25101 entries, 0 to 25100
Data columns (total 52 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   client_id                               25101 non-null  int64  
 1   target                                  25101 non-null  int64  
 2   age                                     25101 non-null  int64  
 3   job-status                              25101 non-null  int64  
 4   retrirement-status                      25101 non-null  int64  
 5   gender                                  25101 non-null  int64  
 6   children                                25101 non-null  int64  
 7   dependants                              25101 non-null  int64  
 8   education                               25101 non-null  object 
 9   marital-status                          25101 non-null  object 
 10  industry                                22842 non-null  ob

In [6]:
train_data.describe()

,client_id,target,age,job-status,retrirement-status,gender,children,dependants,personal-income,reg-and-fact-equality,...,reg-phone,work-phone,loans,paid-loans,total-payments,total-of-delinquencies,max-delinquency-no,mean-delinquency-amount,max-delinquency-amount,previous-cards
count,2.510100e+04,25101.000000,25101.000000,25101.000000,25101.000000,25101.000000,25101.000000,25101.000000,2.510100e+04,25101.000000,...,25101.000000,25101.000000,25101.000000,25101.000000,25101.000000,25101.000000,25101.000000,25101.000000,25101.000000,487.000000
mean,1.000514e+06,0.120314,40.384805,0.909127,0.133740,0.654157,1.092905,0.643719,1.435466e+04,0.836819,...,0.057169,0.909884,1.391140,0.756504,7.891757,0.190152,0.127007,231.677167,235.602035,1.008214
std,8.707073e+03,0.325335,11.615202,0.287434,0.340379,0.475652,0.988124,0.808145,8.244362e+04,0.369538,...,0.232170,0.286353,0.801786,0.992758,8.017568,0.675390,0.350361,797.830215,809.160352,0.090348
min,9.854680e+05,0.000000,21.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.400000e+01,0.000000,...,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,9.929560e+05,0.000000,30.000000,1.000000,0.000000,0.000000,0.000000,0.000000,8.000000e+03,1.000000,...,0.000000,1.000000,1.000000,0.000000,4.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,1.000545e+06,0.000000,39.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.200000e+04,1.000000,...,0.000000,1.000000,1.000000,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,1.008049e+06,0.000000,50.000000,1.000000,0.000000,1.000000,2.000000,1.000000,1.700000e+04,1.000000,...,0.000000,1.000000,2.000000,1.000000,8.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,1.015599e+06,1.000000,67.000000,1.000000,1.000000,1.000000,10.000000,7.000000,1.300000e+07,1.000000,...,1.000000,1.000000,11.000000,11.000000,114.000000,13.000000,8.000000,28012.200000,28012.200000,2.000000


In [7]:
print(train_data.isna().sum())

client_id                                     0
target                                        0
age                                           0
job-status                                    0
retrirement-status                            0
gender                                        0
children                                      0
dependants                                    0
education                                     0
marital-status                                0
industry                                   2259
title                                      2259
tp-state                                   2259
tp-foreign                                 2255
job-type                                   2259
family-income                                 0
personal-income                               0
registration-region                           0
fact-region                                   0
postal-region                                 0
last-loan-region                        

In [9]:
# Отделение признаков от целевой переменной
X = train_data.drop(columns=['client_id', 'target'])
y = train_data['target']

In [10]:
# Обработка тестовых данных
X_test = test_data.drop(columns=['client_id'])

In [11]:
# Кодирование категориальных переменных
X_encoded = pd.get_dummies(X, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

In [12]:
# Убедитесь, что у обеих выборок одинаковые признаки
X_encoded, X_test_encoded = X_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

In [13]:
# Заполнение отсутствующих значений
imputer = SimpleImputer(strategy='mean')
X_encoded = imputer.fit_transform(X_encoded)
X_test_encoded = imputer.transform(X_test_encoded)

In [14]:
# Масштабирование данных
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

In [15]:
# Разделение данных на обучающую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [16]:
# Настройка гиперпараметров с помощью GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='roc_auc')
grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)

Лучшие параметры: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}


In [29]:
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [None, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}


In [ ]:
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='roc_auc')
grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)

In [ ]:
print("Лучшие параметры:", grid_search.best_params_)

# Обучение модели с лучшими параметрами
model = grid_search.best_estimator_
model.fit(X_train, y_train)

# Оценка на валидационной выборке
y_val_pred = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_val_pred)
print(f'ROC AUC на валидационной выборке: {roc_auc}')

# Прогноз на тестовых данных
y_test_pred_probs = model.predict_proba(X_test_scaled)[:, 1]

# Преобразование вероятностей в классы (0 или 1)
y_test_pred = (y_test_pred_probs >= 0.5).astype(int)

# Формирование выходного файла
submission = pd.DataFrame({'client-id': test_data['client-id'], 'target': y_test_pred})
submission.to_csv('response_submission.csv', index=False)

In [27]:
# Обучение модели с лучшими параметрами
model = grid_search.best_estimator_
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_split=5, n_estimators=200,
                       random_state=42)

In [28]:
# Оценка на валидационной выборке
y_val_pred = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_val_pred)
print(f'ROC AUC на валидационной выборке: {roc_auc}')

ROC AUC на валидационной выборке: 0.701111582094651


In [19]:
# Прогноз на тестовых данных
y_test_pred_probs = model.predict_proba(X_test_scaled)[:, 1]

In [20]:
# Преобразование вероятностей в классы (0 или 1)
y_test_pred = (y_test_pred_probs >= 0.5).astype(int)

In [22]:
# Формирование выходного файла
submission = pd.DataFrame({'client_id': test_data['client_id'], 'target': y_test_pred})
submission.to_csv('response_submission.csv', index=False)

In [23]:
from google.colab import files

# Выкачивание файла
files.download('response_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>